# Check evidence packets

Change `packet_id`, then restart the kernel and run all cells.
Run the setup cell first; later cells use its variables.
Nothing here changes files or calls a model.

Sources are extracted from PDFs in `private/pdfs/` into `.txt` and `.json` files in `private/sources/`. The JSON records the PDF filename, hashes, extraction details, and page spans. Apple uses `SRC_01`; Microsoft uses `SRC_02`.

In [6]:
import json
from pathlib import Path
from decimal import Decimal
from pprint import pprint

# Find the repo even when Jupyter starts in src/ or a notebook folder.
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/benchmark/private/packets").is_dir()), None)
if repo is None:
    raise FileNotFoundError("Open this notebook from inside the agent_test repository.")
folder = repo / "data/benchmark"
packet_id = "EP_0007"  # EP_0001 = Apple, EP_0002 = Microsoft

packet = json.loads((folder / f"private/packets/{packet_id}.json").read_text(encoding="utf-8"))
masked_packet = json.loads((folder / f"public/evidence_packets/{packet_id}.json").read_text(encoding="utf-8"))
source_id = packet["source_id"]
source = (folder / f"private/sources/{source_id}.txt").read_text(encoding="utf-8")
source_info = json.loads((folder / f"private/sources/{source_id}.json").read_text(encoding="utf-8"))

print("Company:", packet["company"]["name"])
print("Original source:", source_info.get("input_file") or source_info.get("source_url"))
print("Saved text:", folder / f"private/sources/{source_id}.txt")

Company: Amazon.com, Inc.
Original source: private/pdfs/amzn-20251231.pdf
Saved text: /Users/claudia/Desktop/dsa4213/dsa4213 proj/DSA4213_Group11/data/benchmark/private/sources/SRC_07.txt


## The evidence packet

`private` has the original names and source references. `public` has the masked version the agents will see.

In [7]:
pprint(packet, sort_dicts=False)

{'schema_version': '0.1',
 'packet_id': 'EP_0007',
 'company_alias': 'COMPANY_07',
 'company': {'name': 'Amazon.com, Inc.', 'ticker': 'AMZN'},
 'fiscal_year': 2025,
 'filing': {'form': '10-K',
            'accession_number': '0001018724-26-000004',
            'filing_date': '2026-02-06',
            'source_url': 'https://www.sec.gov/Archives/edgar/data/1018724/000101872426000004/amzn-20251231.htm'},
 'source_id': 'SRC_07',
 'evidence': [{'evidence_id': 'EP_0007_E01',
               'section': 'Item 7',
               'text': 'Net Sales\n'
                       'Net sales include product and service sales. Product '
                       'sales represent revenue from the sale of products and '
                       'related shipping fees and digital media content\n'
                       'where we record revenue gross. Service sales primarily '
                       'represent third-party seller fees, which includes '
                       'commissions and any related fulfillmen

## Source text vs stored evidence

`source_span` is `[start, end]` in the saved source text. The end character is excluded.
Each comparison should print `Equal: True`. Use the printed locator to check the original filing too.

In [8]:
for evidence in packet["evidence"]:
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]

    print("\n---", evidence["evidence_id"], "---")
    print("Location:", evidence["locator"])
    print("Source span:", [start, end])
    print("\nFROM SOURCE:\n", source_excerpt)
    print("\nIN PACKET:\n", evidence["text"])
    print("\nEqual:", source_excerpt == evidence["text"])


--- EP_0007_E01 ---
Location: PDF p. 24; Net Sales; full table with fiscal-year and unit headers and sales growth discussion; printed pp. 24-25
Source span: [111008, 113456]

FROM SOURCE:
 Net Sales
Net sales include product and service sales. Product sales represent revenue from the sale of products and related shipping fees and digital media content
where we record revenue gross. Service sales primarily represent third-party seller fees, which includes commissions and any related fulfillment and shipping
fees, AWS sales, advertising services, Amazon Prime membership fees, and certain digital media content subscriptions. Net sales information is as follows (in
millions):
  Year Ended December 31,
 2024 2025
Net Sales:
North America $ 387,497 $ 426,305 
International 142,906 161,894 
AWS 107,556 128,725 
Consolidated $ 637,959 $ 716,924 
Year-over-year Percentage Growth:
North America 10 % 10 %
International 9 13 
AWS 19 20 
Consolidated 11 12 
Year-over-year Percentage Growth, exclud

## Source numbers vs stored numbers

Find each recorded source number in the original excerpt and compare it with the stored value.
Also check the printed context: a matching number alone does not prove it belongs to the right row, year, or unit.

In [9]:
for fact in packet["facts"]:
    evidence = next(e for e in packet["evidence"] if e["evidence_id"] == fact["evidence_id"])
    start, end = evidence["source_span"]
    source_excerpt = source[start:end]
    position = source_excerpt.find(fact["source_value"])
    source_number = source_excerpt[position:position + len(fact["source_value"])] if position >= 0 else None

    print("\n---", fact["fact_id"], fact["metric"], "---")
    print("Evidence:", fact["evidence_id"], "|", fact["locator"])
    print("Period:", fact["period"], "| Unit:", fact["unit"])
    if position >= 0:
        print("Source context:\n", source_excerpt[max(0, position - 100):position + len(fact["source_value"]) + 100])
    print("Source number:", source_number)
    print("Packet number:", fact["value"])
    equal = source_number is not None and Decimal(source_number.replace(",", "")) == Decimal(fact["value"])
    print("Equal:", equal)


--- EP_0007_F01 net_sales ---
Evidence: EP_0007_E01 | Consolidated row; FY2025 column; net_sales
Period: {'label': 'FY2025', 'start': '2025-01-01', 'end': '2025-12-31'} | Unit: USD_million
Source context:
 a $ 387,497 $ 426,305 
International 142,906 161,894 
AWS 107,556 128,725 
Consolidated $ 637,959 $ 716,924 
Year-over-year Percentage Growth:
North America 10 % 10 %
International 9 13 
AWS 19 20 
Consolida
Source number: 716,924
Packet number: 716924
Equal: True

--- EP_0007_F02 net_sales ---
Evidence: EP_0007_E01 | Consolidated row; FY2024 column; net_sales
Period: {'label': 'FY2024', 'start': '2024-01-01', 'end': '2024-12-31'} | Unit: USD_million
Source context:
 rth America $ 387,497 $ 426,305 
International 142,906 161,894 
AWS 107,556 128,725 
Consolidated $ 637,959 $ 716,924 
Year-over-year Percentage Growth:
North America 10 % 10 %
International 9 13 
AWS 19 20 
Source number: 637,959
Packet number: 637959
Equal: True

--- EP_0007_F03 operating_income ---
Evidence: EP_0007_

## Original vs masked

Read each pair. Names and brands should change; the numbers and meaning should stay the same.

In [10]:
for evidence in packet["evidence"]:
    masked = next(e for e in masked_packet["evidence"] if e["evidence_id"] == evidence["evidence_id"])
    print("\n---", evidence["evidence_id"], "---")
    print("ORIGINAL:\n", evidence["text"])
    print("\nMASKED:\n", masked["text"])

print("\nStored numbers unchanged by masking:")
for fact in packet["facts"]:
    masked_fact = next(f for f in masked_packet["facts"] if f["fact_id"] == fact["fact_id"])
    print(fact["fact_id"], fact["value"], "->", masked_fact["value"], "Equal:", fact["value"] == masked_fact["value"])


--- EP_0007_E01 ---
ORIGINAL:
 Net Sales
Net sales include product and service sales. Product sales represent revenue from the sale of products and related shipping fees and digital media content
where we record revenue gross. Service sales primarily represent third-party seller fees, which includes commissions and any related fulfillment and shipping
fees, AWS sales, advertising services, Amazon Prime membership fees, and certain digital media content subscriptions. Net sales information is as follows (in
millions):
  Year Ended December 31,
 2024 2025
Net Sales:
North America $ 387,497 $ 426,305 
International 142,906 161,894 
AWS 107,556 128,725 
Consolidated $ 637,959 $ 716,924 
Year-over-year Percentage Growth:
North America 10 % 10 %
International 9 13 
AWS 19 20 
Consolidated 11 12 
Year-over-year Percentage Growth, excluding the effect of foreign exchange rates:
North America 10 % 10 %
International 10 10 
AWS 19 20 
Consolidated 11 12 
Net Sales Mix:
North America 61 % 59 %
I

## Notes

Keep any problems found here in the matching report review notebook,
for example `private/reviews/EP_0001_review.ipynb`.
That one notebook holds the evidence checks, report feedback, edits and final output.